# SKB-Q | FINAL one-shot, frozen Qwen evaluation (2026-10-20)

**Research status:** Frozen experiment, not a successful result. **Do not** adjust maps, alpha, CVaR parameters, data prefix, quantizer, baselines, scoring, or analysis after outcomes are known. Submit the complete result ZIP regardless of winner.

- Model: `Qwen/Qwen2.5-1.5B`, fixed commit `8faed761d45a263340a0528343f099c05c9a4323`.
- Training: **30** C4 development maps, all selectors trained from the same maps; risk selector also uses **blockwise NLL targets**. No new training of foundation model.
- Fixed candidate allocation size **K=14** out of 28 attention groups; all evaluated Linear weights simulated 4-bit RTN except selected attention groups 8-bit; nominal mean 4.199719781 bits.
- Untouched target section: WikiText-2 **test** blocks **64–127**, 64 non-overlapping 1024-token blocks (65,472 scored target positions). Earlier published target results used blocks **0–63**. These sections need not be document-independent, and the method was developed after earlier results.
- Methods: `RiskCVaR_30`, `BlockRidgeMean_30`, `Ridge01_30`, `LassoCV_30`, `OLS_30_CASQ_family`, `BestDevR21_30`, `Late_30`.
- **Important:** OLS is an additive CASQ-family proxy, **not a faithful complete CASQ reproduction**; success on one target section is not general method superiority or journal acceptance.

Before using Run All, upload `SKBQ_Qwen_FINAL_FROZEN_2026-10-20.json` next to this notebook (or set `SKBQ_FREEZE_MANIFEST` to its exact path). Activate a Kaggle/Colab GPU with enough memory; do not modify the manifest. All outputs go under `/kaggle/working/SKBQ_Qwen_Final_Locked_Trial` or `/mnt/data/...` on Colab. Default **FULL** evaluates all seven maps; `SKBQ_RUN_MODE=DRY_RUN` checks the model/data locks without inference. Frozen data/code identification is checked before any losses are evaluated.


## 0. Pin runtime dependencies (match original Qwen RTN notebook)

In [ ]:
import sys,subprocess,importlib.metadata as meta
wanted={'transformers':'4.49.0','datasets':'3.5.1','huggingface-hub':'0.36.2','safetensors':'0.5.3'}
missing=[]
for pkg,version in wanted.items():
    try: now=meta.version(pkg)
    except meta.PackageNotFoundError: now=None
    if now!=version:missing.append(f'{pkg}=={version}')
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
    print('Pinned dependencies installed. Restart runtime if a package was already imported with another version.')
else: print('Dependencies pinned')

## 1. Setup and immutable manifest verification

In [ ]:
import os, json, hashlib, shutil, math, gc, random, platform
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

RUN_MODE=os.environ.get('SKBQ_RUN_MODE','FULL').upper()  # Set DRY_RUN for model/dataset checks only
assert RUN_MODE in ('FULL','DRY_RUN')
OUT=Path('/kaggle/working/SKBQ_Qwen_Final_Locked_Trial') if Path('/kaggle/working').exists() else Path('/mnt/data/SKBQ_Qwen_Final_Locked_Trial')
OUT.mkdir(parents=True,exist_ok=True)
assert torch.cuda.is_available(), 'Activate GPU T4 in Kaggle Notebook Settings'
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False

def canon(obj):return json.dumps(obj,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False).encode('utf8')
def sha(obj):return hashlib.sha256(obj).hexdigest()
def save(name,data):
    path=OUT/name; tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf8')
    tmp.replace(path)
print('GPU:',torch.cuda.get_device_name(0),'torch:',torch.__version__,'transformers:',__import__('transformers').__version__)

In [ ]:
FROZEN_MANIFEST_PATH=Path(os.environ.get('SKBQ_FREEZE_MANIFEST','SKBQ_Qwen_FINAL_FROZEN_2026-10-20.json'))
if not FROZEN_MANIFEST_PATH.is_file():
    raise FileNotFoundError('Upload the exact frozen JSON next to this notebook or set SKBQ_FREEZE_MANIFEST.')
FROZEN=json.loads(FROZEN_MANIFEST_PATH.read_text(encoding='utf-8'))
manifest_sha=FROZEN['manifest_sha256']
EXPECTED_SHA='188866e4c113a60cbee231473984aa52b2b8bcc1399269cb803a481f532a63a6'
assert manifest_sha==EXPECTED_SHA, 'Wrong frozen manifest: STOP'
assert sha(canon({k:v for k,v in FROZEN.items() if k!='manifest_sha256'}))==manifest_sha, 'Manifest modified: STOP'
assert FROZEN['status']=='MAPS_AND_ANALYSIS_FROZEN_BEFORE_NEW_TARGET_OUTCOMES; GPU_RUN_PENDING'
ORDER=list(FROZEN['maps'])
assert ORDER==['RiskCVaR_30','BlockRidgeMean_30','Ridge01_30','LassoCV_30',
              'OLS_30_CASQ_family','BestDevR21_30','Late_30']
assert len({tuple(v) for v in FROZEN['maps'].values()})==7
assert all(len(v)==len(set(v))==14 and min(v)>=0 and max(v)<28 for v in FROZEN['maps'].values())
assert all(v==sorted(v) for v in FROZEN['maps'].values())
assert FROZEN['target_corpus']['block_start_index']==64
assert FROZEN['target_corpus']['nonoverlapping_blocks']==64
assert FROZEN['model']['revision']=='8faed761d45a263340a0528343f099c05c9a4323'
assert all(sha(canon(v))==FROZEN['map_sha256'][n] for n,v in FROZEN['maps'].items())
save('FROZEN_MANIFEST_COPIED_BEFORE_EVALUATION.json',FROZEN)
print('Frozen trial VERIFIED:',manifest_sha,' / Policies:',ORDER)


## 2. Load exact model, verify architecture, weights and exact nominal bit budget

In [ ]:
MODEL=FROZEN['model']; PIN=MODEL['revision'];MODEL_ID=MODEL['repo_id']
tok=AutoTokenizer.from_pretrained(MODEL_ID,revision=PIN,trust_remote_code=False)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=PIN,torch_dtype=torch.float16,
                       attn_implementation='sdpa',trust_remote_code=False,low_cpu_mem_usage=True).eval().to('cuda')
model.config.use_cache=False
cfg=model.config
assert cfg.model_type=='qwen2' and cfg.num_hidden_layers==28 and cfg.hidden_size==1536
assert cfg.num_attention_heads==12 and cfg.num_key_value_heads==2
if model.lm_head.weight.data_ptr()==model.get_input_embeddings().weight.data_ptr():
    model.lm_head.weight=nn.Parameter(model.lm_head.weight.detach().clone(),requires_grad=False)
assert model.lm_head.weight.data_ptr()!=model.get_input_embeddings().weight.data_ptr()
embedding_pre=sha(model.get_input_embeddings().weight.detach().cpu().numpy().tobytes())
assert embedding_pre==MODEL['input_embedding_sha256_before_quantization'], 'Embedding fingerprint mismatch: STOP'
modules={n:m for n,m in model.named_modules() if isinstance(m,nn.Linear)}
expected={f'model.layers.{i}.self_attn.{p}' for i in range(28) for p in ('q_proj','k_proj','v_proj','o_proj')}
expected|={f'model.layers.{i}.mlp.{p}' for i in range(28) for p in ('gate_proj','up_proj','down_proj')}
expected.add('lm_head')
assert set(modules)==expected, 'Unexpected Linear inventory'
WEIGHT_COUNT=sum(m.weight.numel() for m in modules.values())
GROUP_WEIGHT=sum(modules[f'model.layers.0.self_attn.{p}'].weight.numel() for p in ('q_proj','k_proj','v_proj','o_proj'))
assert all(sum(modules[f'model.layers.{i}.self_attn.{p}'].weight.numel() for p in ('q_proj','k_proj','v_proj','o_proj'))==GROUP_WEIGHT for i in range(28))
BUDGET=4+4*14*GROUP_WEIGHT/WEIGHT_COUNT
assert WEIGHT_COUNT==MODEL['accounted_linear_params'] and GROUP_WEIGHT==MODEL['attention_group_params']
assert abs(BUDGET-MODEL['avgbit_linear'])<1e-12
master={n:m.weight.detach().cpu().clone() for n,m in modules.items()}
save('MODEL_AND_BUDGET_LOCK.json',{'model_id':MODEL_ID,'model_sha':PIN,'torch':torch.__version__,
    'transformers':__import__('transformers').__version__,'group_params':GROUP_WEIGHT,
    'linear_params':WEIGHT_COUNT,'avgbit_linear':BUDGET,'embedding_sha256':embedding_pre,
    'manifest_sha256':manifest_sha,'linear_module_names':sorted(modules)})
print('Matched mean-bit proxy:',repr(BUDGET),'modules:',len(modules))

## 3. Lock never-before-evaluated WikiText-2 test token positions before measuring any losses

In [ ]:
EV=FROZEN['target_corpus']
ds=load_dataset(EV['id'],EV['config'],revision=EV['revision'],split=EV['split'])
texts=[str(row['text']).strip() for row in ds if row['text'] and str(row['text']).strip()]
ids=tok('\n\n'.join(texts),add_special_tokens=False)['input_ids']
COUNT=EV['nonoverlapping_blocks'];L=EV['seq_length'];BEGIN=EV['block_start_index']
assert COUNT==64 and L==1024 and BEGIN==64
need=(BEGIN+COUNT)*L
assert len(ids)>=need,f'Dataset too short: {len(ids)} < {need} tokens; do not change the number of blocks'
blocks=np.asarray(ids[BEGIN*L:need],dtype='<i4').reshape(COUNT,L)
token_sha=sha(blocks.tobytes())
save('TARGET_TOKEN_LOCK_BEFORE_EVALUATION.json',{'model_sha':PIN,
    'dataset_id':EV['id'],'config':EV['config'],'revision':EV['revision'],'split':EV['split'],
    'used_rows':len(texts),'available_tokens':len(ids),'evaluated_tokens':COUNT*L,
    'predicted_tokens':COUNT*(L-1),'block_range':[BEGIN,BEGIN+COUNT-1],
    'target_token_sha256':token_sha,'manifest_sha256':manifest_sha})
print('64 unseen target blocks [64..127] locked:',token_sha)
if RUN_MODE=='DRY_RUN':
    print('DRY_RUN complete: no model NLL outcomes evaluated; switch RUN_MODE to FULL and Run All')

## 4. Evaluate seven fully frozen group maps, one checkpoint every 4 blocks

In [ ]:
def rtn_row_chunk(w,bits):
    qmax=(1<<(bits-1))-1
    v=w.to(device='cuda',dtype=torch.float32)
    scale=v.abs().amax(dim=1,keepdim=True)/qmax
    scale=torch.where(scale>0,scale,torch.ones_like(scale))
    return ((v/scale).round().clamp(-qmax,qmax)*scale).to(torch.float16)

def map_bits(selected):
    assert len(selected)==14
    b={n:4 for n in modules}
    for layer in selected:
        for p in ('q_proj','k_proj','v_proj','o_proj'):
            b[f'model.layers.{layer}.self_attn.{p}']=8
    avg=sum(modules[n].weight.numel()*b[n] for n in modules)/WEIGHT_COUNT
    assert abs(avg-BUDGET)<1e-12
    return b,avg

def install_bits(mapping):
    with torch.no_grad():
        for n,m in modules.items():
            original=master[n];bit=mapping[n]
            for start in range(0,original.shape[0],4096):
                end=min(start+4096,original.shape[0]);m.weight.data[start:end].copy_(rtn_row_chunk(original[start:end],bit))

RESULTS=[]
if RUN_MODE=='FULL':
    for pidx,policy in enumerate(ORDER,1):
        selected=FROZEN['maps'][policy]
        mapping,bit=map_bits(selected)
        signature=sha(canon({'model_revision':PIN,'policy':policy,'map':selected,'target_sha':token_sha,
                             'manifest_sha':manifest_sha,'avgbit':bit,'blocks':COUNT}))
        path=OUT/f'checkpoint_{policy}.json';losses=[]
        if path.exists():
            prior=json.loads(path.read_text())
            assert prior['fingerprint']==signature,'Stale checkpoint: STOP'
            losses=list(map(float,prior['block_mean_nll']))
        assert len(losses)<=COUNT
        print(f'[{pidx}/{len(ORDER)}] {policy}, existing {len(losses)}/{COUNT}',flush=True)
        install_bits(mapping)
        for bi in range(len(losses),COUNT):
            x=torch.as_tensor(blocks[bi],dtype=torch.long,device='cuda').unsqueeze(0)
            with torch.inference_mode():
                out=model(input_ids=x,use_cache=False)
                logits=out.logits[:,:-1,:].float().contiguous()
                loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),x[:,1:].reshape(-1))
                value=float(loss.item())
            assert math.isfinite(value)
            losses.append(value)
            del x,out,logits,loss
            if (bi+1)%4==0 or bi==COUNT-1:
                save(f'checkpoint_{policy}.json',{'policy':policy,'selected_layers':selected,'avgbit_linear':bit,
                    'model_revision':PIN,'frozen_manifest_sha256':manifest_sha,'target_token_sha256':token_sha,
                    'allocation_sha256':sha(canon(mapping)),'fingerprint':signature,
                    'block_mean_nll':losses,'blocks_complete':len(losses)})
                print(' ',bi+1,'/',COUNT,'meanNLL',round(float(np.mean(losses)),7),flush=True)
        RESULTS.append({'policy':policy,'mean_nll':float(np.mean(losses)),
            'ppl':float(math.exp(float(np.mean(losses)))),'blocks':COUNT,
            'predicted_tokens':COUNT*(L-1),'avgbit_linear':bit,'allocation_sha256':sha(canon(mapping))})
        save('PROGRESS_SUMMARY.json',RESULTS)
        torch.cuda.empty_cache()
    assert len(RESULTS)==len(ORDER)==7
    assert sha(model.get_input_embeddings().weight.detach().cpu().numpy().tobytes())==embedding_pre
    save('POLICY_RESULTS.json',RESULTS)
    print('Seven policies evaluated. Checkpoints complete.')

## 5. Recompute predeclared decision rule; save losing outcomes too

In [ ]:
if RUN_MODE=='FULL':
    losses={}
    for policy in ORDER:
        p=json.loads((OUT/f'checkpoint_{policy}.json').read_text())
        assert p['policy']==policy and p['selected_layers']==FROZEN['maps'][policy]
        assert p['frozen_manifest_sha256']==manifest_sha and p['target_token_sha256']==token_sha
        assert p['blocks_complete']==64
        losses[policy]=np.array(p['block_mean_nll'],dtype=np.float64)
        assert losses[policy].shape==(64,) and np.isfinite(losses[policy]).all()
    def cvar75(v):
        # Exactly 16 highest-loss blocks of 64
        return float(np.sort(v)[-16:].mean())
    late=losses['Late_30']
    def metrics(policy):
        a=losses[policy]
        return {'nll':float(np.mean(a)),'ppl':float(math.exp(float(a.mean()))),
                'cvar75_excess_vs_late':cvar75(a-late),
                'avgbit_linear':BUDGET,'blocks':64,'scored_tokens':65472}
    outcomes={p:metrics(p) for p in ORDER}
    primary_bases=FROZEN['primary_decision_rule']['comparator_names']
    primary_pass=all(outcomes['RiskCVaR_30']['nll']<outcomes[p]['nll'] for p in primary_bases)
    secondary_bases=FROZEN['secondary_decision_rule']['comparator_names']
    tail_pass=all(outcomes['RiskCVaR_30']['cvar75_excess_vs_late']<outcomes[p]['cvar75_excess_vs_late'] for p in secondary_bases)
    rng=np.random.default_rng(20261020)
    def circular_mbb(d,block_size=8,draws=5000):
        starts=rng.integers(0,64,size=(draws,8))
        idx=(starts[:,:,None]+np.arange(block_size)[None,None,:])%64
        return np.percentile(d[idx.reshape(draws,64)].mean(axis=1),[2.5,97.5]).tolist()
    comparisons={}
    for p in ORDER:
        if p=='RiskCVaR_30':continue
        d=losses['RiskCVaR_30']-losses[p]
        comparisons[p]={'risk_minus_baseline_mean_nll':float(d.mean()),
                        'relative_ppl_reduction_pct':float(100*(1-math.exp(float(d.mean())))),
                        'eight_contiguous_windows_risk_wins':int(sum(float(np.mean(d[i:i+8]))<0 for i in range(0,64,8))),
                        'circular_moving_block_bootstrap_95pct_L8_descriptive':circular_mbb(d)}
    report={'study_id':FROZEN['study_id'],'frozen_manifest_sha256':manifest_sha,
        'input_tokens_sha256':token_sha,'target_prefix_blocks':[64,127],
        'mean_nll_primary_predeclared_pass_both_Lasso_and_OLS':bool(primary_pass),
        'secondary_tail_better_than_all_three':bool(tail_pass),
        'outcomes':outcomes,'risk_comparisons':comparisons,
        'scientific_cautions':['A pass is conditional on one Qwen, one RTN, one token segment, not CASQ algorithm superiority',
        'OLS is CASQ-family only; not full CASQ','Moving block bootstrap is descriptive, not document-independent confidence',
        'Risk method designed after earlier historical test results; no unobserved data used for this segment',
        'Report all negative findings without changing hyperparameters or reusing the segment']}
    save('PREDECLARED_FINAL_ANALYSIS.json',report)
    print('PRIMARY RULE PASS:',primary_pass,' | SECONDARY TAIL PASS:',tail_pass)
    for n,z in outcomes.items(): print(n,'PPL',round(z['ppl'],5),'CVaR',round(z['cvar75_excess_vs_late'],6))


## 6. Deliver the archive unchanged; no optimization after seeing the result

In [ ]:
if RUN_MODE=='FULL':
    archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
    print('COMPLETE: save and return this exact ZIP (all 7 checkpoints):',archive)
    print('Never rerank, tune, or switch the frozen test portion after inspecting outcomes.')
else:
    print('DRY_RUN: only model/token integrity checked. No policy losses. Set SKBQ_RUN_MODE=FULL and run from first cell.')
